## Library imports

In [16]:
# libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import os
from pathlib import Path
from tkinter import Tk
from tkinter.filedialog import askdirectory

# Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Metrics
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_auc_score
from sklearn.metrics import classification_report
from sklearn.model_selection import cross_val_score
from sklearn.metrics import f1_score

# Models
import lightgbm as lgb
from catboost import CatBoostClassifier
from xgboost import XGBClassifier

# Hyperparameter tuning
from sklearn.model_selection import StratifiedKFold
import optuna

# exporting the model
import joblib

## Load the dataset

In [17]:
cleaned_data_path = Path("../streamlit/cleaned_data")
if not cleaned_data_path.exists():
    root = Tk()
    root.withdraw()
    cleaned_data_path = Path(askdirectory(title="Select the directory containing the cleaned data"))

In [18]:
train=pd.read_parquet(cleaned_data_path/'train_final.parquet')
test=pd.read_parquet(cleaned_data_path/'test_final.parquet')

In [19]:
for col in train.columns:
    print(col)

SK_ID_CURR
TARGET
NAME_CONTRACT_TYPE
CODE_GENDER
FLAG_OWN_CAR
FLAG_OWN_REALTY
CNT_CHILDREN
AMT_INCOME_TOTAL
AMT_CREDIT
AMT_ANNUITY
AMT_GOODS_PRICE
NAME_TYPE_SUITE
NAME_INCOME_TYPE
NAME_EDUCATION_TYPE
NAME_FAMILY_STATUS
NAME_HOUSING_TYPE
REGION_POPULATION_RELATIVE
DAYS_BIRTH
DAYS_EMPLOYED
DAYS_REGISTRATION
DAYS_ID_PUBLISH
FLAG_EMP_PHONE
FLAG_WORK_PHONE
FLAG_CONT_MOBILE
FLAG_PHONE
FLAG_EMAIL
OCCUPATION_TYPE
CNT_FAM_MEMBERS
REGION_RATING_CLIENT
REGION_RATING_CLIENT_W_CITY
WEEKDAY_APPR_PROCESS_START
HOUR_APPR_PROCESS_START
REG_REGION_NOT_LIVE_REGION
REG_REGION_NOT_WORK_REGION
LIVE_REGION_NOT_WORK_REGION
REG_CITY_NOT_LIVE_CITY
REG_CITY_NOT_WORK_CITY
LIVE_CITY_NOT_WORK_CITY
ORGANIZATION_TYPE
OBS_30_CNT_SOCIAL_CIRCLE
DEF_30_CNT_SOCIAL_CIRCLE
OBS_60_CNT_SOCIAL_CIRCLE
DEF_60_CNT_SOCIAL_CIRCLE
DAYS_LAST_PHONE_CHANGE
AMT_REQ_CREDIT_BUREAU_HOUR
AMT_REQ_CREDIT_BUREAU_DAY
AMT_REQ_CREDIT_BUREAU_WEEK
AMT_REQ_CREDIT_BUREAU_MON
AMT_REQ_CREDIT_BUREAU_QRT
AMT_REQ_CREDIT_BUREAU_YEAR
EXT_SOURCE_MEAN
HAS_BUI

### target separation

In [20]:
X = train.drop(columns=['SK_ID_CURR', 'TARGET'])
y = train['TARGET']

### endoding categorical features

In [21]:
cat_cols = X.select_dtypes(include='object').columns

len(cat_cols)

0

In [22]:
cat_cols

Index([], dtype='object')

In [23]:
encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    encoders[col] = le

### split the data into training and validation sets

In [24]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2,stratify= y ,random_state=42)

# model testing

### base model testing

In [25]:
model = lgb.LGBMClassifier(
    n_estimators      = 1000,
    learning_rate     = 0.05,
    num_leaves        = 63,
    max_depth         = -1,
    min_child_samples = 50,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    reg_alpha         = 0.1,
    reg_lambda        = 0.1,
    class_weight      = "balanced",   # handles class imbalance
    random_state      = 42,
    n_jobs            = -1,
    verbose           = -1,
)

model.fit(
    X_train, y_train,
    eval_set              = [(X_val, y_val)],
    callbacks             = [lgb.early_stopping(50, verbose=False),
                                lgb.log_evaluation(period=-1)],
)

pred_prob = model.predict_proba(X_val)[:, 1]
auc       = roc_auc_score(y_val, pred_prob)

In [26]:
auc

np.float64(0.7664056260372388)

In [27]:
pred = model.predict(X_val)

confusion_matrix(y_val, pred)

array([[48340,  8165],
       [ 2525,  2437]])

In [28]:
print(classification_report(y_val,pred))

              precision    recall  f1-score   support

           0       0.95      0.86      0.90     56505
           1       0.23      0.49      0.31      4962

    accuracy                           0.83     61467
   macro avg       0.59      0.67      0.61     61467
weighted avg       0.89      0.83      0.85     61467



In [29]:
feature_imp = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': model.feature_importances_
}).sort_values(
    by='Importance',
    ascending=False
)

feature_imp.tail(50)

,Feature,Importance
38,DEF_30_CNT_SOCIAL_CIRCLE,92
13,NAME_HOUSING_TYPE,85
141,AVG_RECEIVABLE,83
135,AVG_CC_DPD,83
4,CNT_CHILDREN,81
22,FLAG_PHONE,81
40,DEF_60_CNT_SOCIAL_CIRCLE,78
9,NAME_TYPE_SUITE,75
45,AMT_REQ_CREDIT_BUREAU_MON,75
20,FLAG_WORK_PHONE,66


In [30]:
cv = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

In [31]:
cv_scores = cross_val_score(model,X,y,scoring='roc_auc',cv=cv,n_jobs=-1)

In [32]:
print(cv_scores)
print(cv_scores.mean())
print(cv_scores.std())

[0.77346374 0.77522504 0.7596798  0.76727879 0.77199067]
0.76952760704354
0.005586718041530355


## hyperparameter tuning with optuna for lightgbm

In [79]:
def objective(trial):
    params={
        'objective': 'binary',
        'metric': 'auc',
        'boosting_type': 'gbdt',
        'class_weight': 'balanced',
        'n_estimators': trial.suggest_int('n_estimators', 300,2000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1,
    }
    model = lgb.LGBMClassifier(**params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(50, verbose=False)],
    )
    pred_prob = model.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, pred_prob)
    return auc

In [ ]:
lgb_study = optuna.create_study(direction='maximize')
lgb_study.optimize(objective, n_trials=50)

[I 2026-05-28 22:06:44,846] A new study created in memory with name: no-name-b9ef83a7-f892-4769-b847-9b8b5842febe
[I 2026-05-28 22:07:15,701] Trial 0 finished with value: 0.7750925973778023 and parameters: {'n_estimators': 1410, 'learning_rate': 0.013291023941685578, 'num_leaves': 88, 'max_depth': 4, 'subsample': 0.7021300592404628, 'colsample_bytree': 0.967149974778681, 'reg_alpha': 0.07634724593613502, 'reg_lambda': 0.17949808061534578}. Best is trial 0 with value: 0.7750925973778023.
[I 2026-05-28 22:07:30,636] Trial 1 finished with value: 0.776491277965257 and parameters: {'n_estimators': 748, 'learning_rate': 0.03543860682357844, 'num_leaves': 27, 'max_depth': 4, 'subsample': 0.6389011108898541, 'colsample_bytree': 0.7231821108536258, 'reg_alpha': 0.0012648186229355647, 'reg_lambda': 0.002629973614091467}. Best is trial 1 with value: 0.776491277965257.
[I 2026-05-28 22:08:00,372] Trial 2 finished with value: 0.7784138480859094 and parameters: {'n_estimators': 1255, 'learning_rate'

In [ ]:
lgb_study.best_params

{'n_estimators': 1756,
 'learning_rate': 0.01331366272815625,
 'num_leaves': 74,
 'max_depth': 12,
 'subsample': 0.8567488572903952,
 'colsample_bytree': 0.6144388426493165,
 'reg_alpha': 9.928108621631159,
 'reg_lambda': 3.2730873799027798}

In [ ]:
lgb_study.best_value

0.7803364396062585

## tunned lightgbm model

In [ ]:
lgb_model = lgb.LGBMClassifier(
    **lgb_study.best_params,
    class_weight      = "balanced",
    random_state      = 42,
    n_jobs            = -1,
    verbose           = -1,
)

In [61]:
lgb_model.fit(
    X_train,
    y_train,
    eval_set=(X_val, y_val)
)

LGBMClassifier(class_weight='balanced', colsample_bytree=0.6144388426493165,
               learning_rate=0.01331366272815625, max_depth=12,
               min_child_samples=50, n_estimators=1756, n_jobs=-1,
               num_leaves=74, random_state=42, reg_alpha=9.928108621631159,
               reg_lambda=3.2730873799027798, subsample=0.8567488572903952,
               verbose=-1)

In [ ]:
lgb_pred = lgb_model.predict_proba(X_val)[:,1]
lgb_auc = roc_auc_score(y_val, lgb_pred)
print(lgb_auc)

0.780350392208285


### hyperparameter tuning with optuna for catboost

In [ ]:
# =========================================
# OPTUNA OBJECTIVE FUNCTION FOR CATBOOST
# =========================================

def objective(trial):

    params = {

        'loss_function': 'Logloss',

        'eval_metric': 'AUC',

        'iterations': trial.suggest_int(
            'iterations',
            300,
            2000
        ),

        'learning_rate': trial.suggest_float(
            'learning_rate',
            0.01,
            0.1,
            log=True
        ),

        'depth': trial.suggest_int(
            'depth',
            4,
            10
        ),

        'l2_leaf_reg': trial.suggest_float(
            'l2_leaf_reg',
            1e-3,
            10.0,
            log=True
        ),

        'subsample': trial.suggest_float(
            'subsample',
            0.6,
            1.0
        ),

        'random_strength': trial.suggest_float(
            'random_strength',
            1e-3,
            10.0,
            log=True
        ),

        'bagging_temperature': trial.suggest_float(
            'bagging_temperature',
            0,
            10
        ),

        'scale_pos_weight': trial.suggest_float(
            'scale_pos_weight',
            1,
            10
        ),

        'random_state': 42,

        'verbose': 0
    }

    # =====================================
    # MODEL
    # =====================================

    cat_model = CatBoostClassifier(
        **params
    )

    # =====================================
    # TRAIN
    # =====================================

    cat_model.fit(

        X_train,
        y_train,

        eval_set=(X_val, y_val),

        early_stopping_rounds=50,

        verbose=False
    )

    # =====================================
    # PREDICTION
    # =====================================

    pred_prob = cat_model.predict_proba(X_val)[:,1]

    # =====================================
    # ROC-AUC
    # =====================================

    auc = roc_auc_score(y_val,pred_prob)

    return auc

In [ ]:
cat_study = optuna.create_study(direction='maximize')

cat_study.optimize(objective,n_trials=75)

[I 2026-05-28 22:53:52,588] A new study created in memory with name: no-name-0676b5b0-4517-4861-aeb4-3a20f8f63e36
[I 2026-05-28 22:54:06,477] Trial 0 finished with value: 0.7714439669815525 and parameters: {'iterations': 1295, 'learning_rate': 0.05906329856521286, 'depth': 8, 'l2_leaf_reg': 0.12978727383872166, 'subsample': 0.7273884719359953, 'random_strength': 0.03289372924152347, 'bagging_temperature': 5.059569565681148, 'scale_pos_weight': 9.107935794226211}. Best is trial 0 with value: 0.7714439669815525.
[I 2026-05-28 22:54:24,291] Trial 1 finished with value: 0.7775268271051835 and parameters: {'iterations': 866, 'learning_rate': 0.056992785034403644, 'depth': 5, 'l2_leaf_reg': 0.3922040849042633, 'subsample': 0.6006743679474543, 'random_strength': 0.02993362187767755, 'bagging_temperature': 7.510896274521298, 'scale_pos_weight': 8.7770499693569}. Best is trial 1 with value: 0.7775268271051835.
[I 2026-05-28 22:55:12,934] Trial 2 finished with value: 0.7646618789126002 and param

In [ ]:
cat_study.best_params

{'iterations': 1825,
 'learning_rate': 0.03480089433662196,
 'depth': 5,
 'l2_leaf_reg': 9.975131167809268,
 'subsample': 0.9944006706987512,
 'random_strength': 1.0556500215634455,
 'bagging_temperature': 4.6402150694429505,
 'scale_pos_weight': 5.255271095208505}

In [ ]:
cat_study.best_value

0.7821605461573439

In [ ]:
# =========================================
# FINAL TUNED CATBOOST MODEL
# =========================================
cat_model = CatBoostClassifier(
    **cat_study.best_params,
    loss_function='Logloss',
    eval_metric='AUC',
    random_state=42,
    verbose=100
)

# =========================================
# TRAIN MODEL
# =========================================

cat_model.fit(

    X_train,
    y_train,

    eval_set=(X_val, y_val),

    early_stopping_rounds=50,

    verbose=False
)

# =========================================
# PREDICTIONS
# =========================================

cat_pred = cat_model.predict_proba(X_val)[:,1]

# =========================================
# ROC-AUC SCORE
# =========================================

cat_auc = roc_auc_score(y_val,cat_pred)

print("CatBoost ROC-AUC :", cat_auc)

CatBoost ROC-AUC : 0.7821605461573439


In [40]:
feature_imp = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': cat_model.feature_importances_
}).sort_values(
    by='Importance',
    ascending=False
)

feature_imp.tail(50)

,Feature,Importance
45,AMT_REQ_CREDIT_BUREAU_MON,0.195066
22,FLAG_PHONE,0.189384
137,CC_LATE_PAYMENT_RATIO,0.179224
59,CLOSED_CREDIT_COUNT,0.175449
37,OBS_30_CNT_SOCIAL_CIRCLE,0.174622
9,NAME_TYPE_SUITE,0.168558
139,MAX_DRAWINGS_CURRENT,0.159432
0,NAME_CONTRACT_TYPE,0.159013
84,OVERDUE_PER_ACTIVE_LOAN,0.151397
89,CANCELLED_COUNT,0.150473


### checking if dropping 0 importance in both models improves performance

In [41]:
col_to_drop = feature_imp[feature_imp['Importance'] == 0]['Feature'].tolist()
X_train_reduced = X_train.drop(columns=col_to_drop)
X_val_reduced = X_val.drop(columns=col_to_drop)

In [42]:
cat_model.fit(

    X_train_reduced,
    y_train,

    eval_set=(X_val_reduced, y_val),

    early_stopping_rounds=50,

    verbose=False
)
cat_pred = cat_model.predict_proba(X_val_reduced)[:,1]
cat_auc = roc_auc_score(y_val,cat_pred)

print("CatBoost ROC-AUC :", cat_auc)

CatBoost ROC-AUC : 0.7813902355539477


### instead of helping it actually made it drop the roc_auc, so we will keep all features for the final model

### ensemble on two models
ensemble_pred = 0.5 * cat_pred + 0.5 * lgb_pred

In [43]:
blend_pred = (
    0.5 * lgb_pred
    +
    0.5 * cat_pred
)

In [ ]:
blend_auc = roc_auc_score(y_val,blend_pred)

print("Blended ROC-AUC :", blend_auc)

Blended ROC-AUC : 0.7824372085651143


### FINAL xgboost model

In [ ]:
# =========================================
# XGBOOST MODEL
# =========================================
xgb_model = XGBClassifier(
    objective='binary:logistic',
    eval_metric='auc',
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    scale_pos_weight=(
        y_train.value_counts()[0]
        /
        y_train.value_counts()[1]
    ),
    random_state=42,
    n_jobs=-1
)

# =========================================
# TRAIN XGBOOST
# =========================================

xgb_model.fit(

    X_train,
    y_train,

    eval_set=[(X_val, y_val)],

    verbose=False
)

# =========================================
# XGBOOST PREDICTIONS
# =========================================

xgb_pred = xgb_model.predict_proba(X_val)[:,1]

# =========================================
# XGBOOST ROC-AUC
# =========================================

xgb_auc = roc_auc_score(y_val,xgb_pred)

print("XGBoost ROC-AUC :", xgb_auc)

XGBoost ROC-AUC : 0.7706375461738573


In [ ]:
# =========================================
# TRIPLE ENSEMBLE
# =========================================

ensemble_pred_prob = (
    0.5 * cat_pred
    +
    0.3 * lgb_pred
    +
    0.2 * xgb_pred
)

# =========================================
# ENSEMBLE ROC-AUC
# =========================================

ensemble_auc = roc_auc_score(y_val,ensemble_pred_prob)

print("Triple Ensemble ROC-AUC :", ensemble_auc)

Triple Ensemble ROC-AUC : 0.7824745011026373


In [50]:
ensemble_pred = (ensemble_pred_prob >= 0.5).astype(int)


In [51]:
print(classification_report(y_val,ensemble_pred))

              precision    recall  f1-score   support

           0       0.95      0.86      0.91     56505
           1       0.24      0.50      0.33      4962

    accuracy                           0.83     61467
   macro avg       0.60      0.68      0.62     61467
weighted avg       0.89      0.83      0.86     61467



In [52]:
print(confusion_matrix(y_val,ensemble_pred))

[[48801  7704]
 [ 2472  2490]]


### threshold tuning

In [53]:
thresholds = np.arange(0.10, 0.91, 0.01)

results = []

for threshold in thresholds:

    pred = (ensemble_pred_prob >= threshold).astype(int)
    f1 = f1_score(y_val,pred)
    results.append([threshold, f1])

results = pd.DataFrame(
    results,
    columns=['Threshold', 'F1']
)

results.sort_values(by='F1',ascending=False).head(10)

,Threshold,F1
45,0.55,0.335284
44,0.54,0.334817
46,0.56,0.334301
47,0.57,0.334139
43,0.53,0.333771
42,0.52,0.332065
49,0.59,0.332058
48,0.58,0.331954
41,0.51,0.331062
50,0.60,0.330301


In [54]:
best_threshold = 0.55

final_pred = (ensemble_pred_prob >= best_threshold).astype(int)

print(classification_report(y_val,final_pred))
print(confusion_matrix(y_val,final_pred))

              precision    recall  f1-score   support

           0       0.95      0.90      0.92     56505
           1       0.27      0.43      0.34      4962

    accuracy                           0.86     61467
   macro avg       0.61      0.67      0.63     61467
weighted avg       0.89      0.86      0.88     61467

[[50802  5703]
 [ 2814  2148]]


### final model comparison

In [56]:
comparison = pd.DataFrame({

    'Model':[
        'LightGBM',
        'CatBoost',
        'XGBoost',
        'Triple Ensemble'
    ],

    'ROC_AUC':[
        lgb_auc,
        cat_auc,
        xgb_auc,
        ensemble_auc
    ]
})

comparison.sort_values(
    by='ROC_AUC',
    ascending=False
)

,Model,ROC_AUC
3,Triple Ensemble,0.782475
1,CatBoost,0.781390
0,LightGBM,0.780350
2,XGBoost,0.770638


# Retrain models on full training data

In [ ]:
lgb_final = lgb.LGBMClassifier(
    n_estimators      = 1756,
    learning_rate     = 0.01331366272815625,
    num_leaves        = 74,
    max_depth         = 12,
    min_child_samples = 50,
    subsample         = 0.8567488572903952,
    colsample_bytree  = 0.6144388426493165,
    reg_alpha         = 9.928108621631159,
    reg_lambda        = 3.2730873799027798,
    class_weight      = "balanced",   # handles class imbalance
    random_state      = 42,
    n_jobs            = -1,
    verbose           = -1,
)

cat_final = CatBoostClassifier(
    loss_function='Logloss',
    eval_metric='AUC',
    iterations=1825,
    learning_rate=0.03480089433662196,
    depth=5,
    l2_leaf_reg=9.975131167809268,
    subsample=0.9944006706987512,
    random_strength=1.0556500215634455,
    bagging_temperature=4.6402150694429505,
    scale_pos_weight=5.255271095208505,
    random_state=42,
    verbose=100
)

xgb_final = XGBClassifier(
    objective='binary:logistic',
    eval_metric='auc',
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    scale_pos_weight=(
        y.value_counts()[0]
        /
        y.value_counts()[1]
    ),
    random_state=42,
    n_jobs=-1
)

In [66]:
cat_final.fit(X, y,verbose=False)
lgb_final.fit(X, y)
xgb_final.fit(X, y)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='auc', feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=1000,
              n_jobs=-1, num_parallel_tree=None, ...)

In [70]:
ensemble_info = {

    'cat_weight': 0.5,

    'lgb_weight': 0.3,

    'xgb_weight': 0.2,

    'threshold': 0.5
}

In [ ]:
model_metrics = {
    "roc_auc": ensemble_auc,
    "cat_weight": 0.5,
    "lgb_weight": 0.3,
    "xgb_weight": 0.2,
    "threshold": 0.5
}

## Exporting the models ,encoders and ensemble info

In [ ]:
model_export_path = Path("../streamlit/models")
if not model_export_path.exists():
    root = Tk()
    root.withdraw()
    model_export_path = Path(askdirectory(title="Select the directory containing the cleaned data"))

In [ ]:
joblib.dump(cat_final,model_export_path/'catboost_model.pkl')

joblib.dump(lgb_final,model_export_path/'lightgbm_model.pkl')

joblib.dump(xgb_final,model_export_path/'xgboost_model.pkl')

joblib.dump(encoders,model_export_path/'encoders.pkl')

joblib.dump(X.columns.tolist(),model_export_path/'feature_columns.pkl')

joblib.dump(ensemble_info,model_export_path/'ensemble_info.pkl')

joblib.dump(model_metrics,model_export_path / "model_metrics.pkl")

['../streamlit/models/ensemble_info.pkl']